In [3]:
import os

print(os.listdir())

['.config', 'diabetes.csv', 'sample_data']


In [4]:
import pandas as pd

df = pd.read_csv("diabetes.csv")

df.head()


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [5]:
print(df.columns)

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')


In [6]:
print(os.listdir())
print(df.columns)

['.config', 'diabetes.csv', 'sample_data']
Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')


In [7]:
# ==========================================================
# STEP 1 : DATA PREPROCESSING
# ==========================================================

import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

print("Original Dataset Shape:", df.shape)

print("\nFirst 5 Rows")
print(df.head())

# Check missing values
print("\nMissing Values Before Preprocessing")
print(df.isnull().sum())

# Replace invalid 0 values with NaN
columns = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

df[columns] = df[columns].replace(0, np.nan)

print("\nMissing Values After Replacing 0 with NaN")
print(df.isnull().sum())

# Fill missing values using Mean
imputer = SimpleImputer(strategy='mean')
df[columns] = imputer.fit_transform(df[columns])

print("\nMissing Values After Mean Imputation")
print(df.isnull().sum())

# Separate Features and Target
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

# Feature Scaling
scaler = StandardScaler()
X = scaler.fit_transform(X)

print("\nData Preprocessing Completed Successfully!")

Original Dataset Shape: (768, 9)

First 5 Rows
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  

Missing Values Before Preprocessing
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age      

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from mlxtend.feature_selection import SequentialFeatureSelector as SFS

# Split Dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Create Classifier
knn = KNeighborsClassifier(n_neighbors=5)

# Sequential Forward Selection
forward = SFS(
    knn,
    k_features=5,
    forward=True,
    floating=False,
    scoring='accuracy',
    cv=5
)

forward.fit(X_train, y_train)

print("========================================")
print("Sequential Forward Selection (SFS)")
print("========================================")

print("\nSelected Features:")
selected_forward = list(forward.k_feature_names_)
print(selected_forward)

# Train Model using selected features
X_train_forward = X_train[:, list(map(int, selected_forward))]
X_test_forward = X_test[:, list(map(int, selected_forward))]

knn.fit(X_train_forward, y_train)

prediction = knn.predict(X_test_forward)

forward_accuracy = accuracy_score(y_test, prediction)

print("\nForward Selection Accuracy:", round(forward_accuracy,4))

Sequential Forward Selection (SFS)

Selected Features:
['1', '2', '3', '5', '7']

Forward Selection Accuracy: 0.7532


In [9]:
# Sequential Backward Selection
backward = SFS(
    knn,
    k_features=5,
    forward=False,
    floating=False,
    scoring='accuracy',
    cv=5
)

backward.fit(X_train, y_train)

print("\n========================================")
print("Sequential Backward Selection (SBS)")
print("========================================")

print("\nSelected Features:")
selected_backward = list(backward.k_feature_names_)
print(selected_backward)

# Train Model using selected features
X_train_backward = X_train[:, list(map(int, selected_backward))]
X_test_backward = X_test[:, list(map(int, selected_backward))]

knn.fit(X_train_backward, y_train)

prediction = knn.predict(X_test_backward)

backward_accuracy = accuracy_score(y_test, prediction)

print("\nBackward Selection Accuracy:", round(backward_accuracy,4))

print("\n========================================")
print("Comparison")
print("========================================")
print("Forward Selection Accuracy :", round(forward_accuracy,4))
print("Backward Selection Accuracy:", round(backward_accuracy,4))


Sequential Backward Selection (SBS)

Selected Features:
['0', '1', '4', '5', '7']

Backward Selection Accuracy: 0.7229

Comparison
Forward Selection Accuracy : 0.7532
Backward Selection Accuracy: 0.7229
